In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


In [ ]:
import torch

new_tokens = [
    # --- Structural & Logic Tags ---
    "I[", "]", "|","INTENT_DETECTOR",
    
    # --- Symbolic Operators & Status Markers ---
     "∷", "$", "GB", "UNLIGB",  "CHITCHAT", "NIC", "BYE", 
     "∈",".F","p.F","d.F", "?(", ")", "[0]", "[*]", ",","&", ">", "<", ">=", "<=", "!=", 
    
    # --- Catalog Identifiers (p01-p58, d01-d55, r01-r52) ---
   "p01","p02","p03","p04","p05","p06","p07","p08","p09","p10","p11","p12","p13","p14","p15","p16","p17","p18","p19",
    "d01","d02","d03","d04","d05","d06","d07","d08","d09","d10","d11","d12","d13","d14","d15","d16","d17","d18","d19",
    "r01","r02","r03","r04","r05","r06","r07","r08","r09","r10","r11","r12","r13","r14","r15","r16","r17","r18","r19","r20","r21","r22","r23","r24"
]

# Clean duplicates
new_tokens = list(set(new_tokens))

def initialize_new_tokens(model, tokenizer, new_tokens):
    # Get the embedding layer
    embeddings = model.get_input_embeddings()
    # Get the LM head
    lm_head = model.get_output_embeddings()
    
    for token in new_tokens:
        token_id = tokenizer.convert_tokens_to_ids(token)
        # Simple heuristic: average the embeddings of the characters in the token
        # This gives the model a much better "starting point" than random noise
        sub_tokens = tokenizer.tokenize(token[1:] if token.startswith(' ') else token)
        sub_token_ids = tokenizer.convert_tokens_to_ids(sub_tokens)
        
        if len(sub_token_ids) > 0:
            with torch.no_grad():
                avg_emb = embeddings.weight[sub_token_ids].mean(dim=0)
                embeddings.weight[token_id] = avg_emb
                
                avg_lm = lm_head.weight[sub_token_ids].mean(dim=0)
                lm_head.weight[token_id] = avg_lm

# Call this after resizing but BEFORE training
tokenizer.add_tokens(new_tokens)
model.resize_token_embeddings(len(tokenizer))
# model.resize_output_embeddings(len(tokenizer))
initialize_new_tokens(model, tokenizer, new_tokens)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],

                      # "embed_tokens", "lm_head",], # Add for continual pretraining
    modules_to_save = ["embed_tokens", "lm_head"],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)


In [ ]:
######### data set for SFT

import json
from datasets import Dataset
import random

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{%- endif %}"""

import json
import random
from datasets import Dataset

def prepare_data_set(file_name, tokenizer, max_seq_length):
    sft_data = []

    # 1. Load data
    with open(file_name, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                sft_data.append(json.loads(line))
    
    random.seed(42) 
    random.shuffle(sft_data)
    
    dataset = Dataset.from_dict({
        "messages": [item["messages"] for item in sft_data]
    })

    # 2. Format into ChatML strings
    def formatting_prompts_func(examples):
        output_texts = []
        for i in range(len(examples['messages'])):
            # add_generation_prompt=False is correct here because we want 
            # the full conversation (including the assistant's answer)
            text = tokenizer.apply_chat_template(
                examples['messages'][i], 
                tokenize=False, 
                add_generation_prompt=False
            )
            # Ensure the sequence ends with EOS
            if not text.endswith(tokenizer.eos_token):
                text += tokenizer.eos_token
            output_texts.append(text)
        return { "text" : output_texts }

    # Map the formatting
    dataset = dataset.map(formatting_prompts_func, batched=True)

    # 3. Handle Tokenization
    # Note: We do NOT remove the 'text' column yet. 
    # SFTTrainer needs 'text' to handle completion-only masking.
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_seq_length,
            padding=False,
        )

    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        # Keep 'text' column for Phase 2 collator!
        # Only remove 'messages' as it's a complex dict
        remove_columns=["messages"] 
    )
    
    print(f"Dataset ready with {len(tokenized_dataset)} rows.")
    return tokenized_dataset

In [ ]:
#Phase 1 - SFT error on whole i/p
from unsloth import UnslothTrainer, UnslothTrainingArguments
train_dataset = prepare_data_set("/mnt/data/training_data/intent_det_sft_template__20260407_144442.txt", tokenizer, max_seq_length)
sft_trainer_p1 = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,
    dataset_batched=True,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 1,
        max_steps = 3,
        learning_rate = 5e-5,
        logging_steps = 5,
        # save_steps = 50,
        # save_total_limit=2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/mnt/data/outputs/",
        report_to = "none",
        
    ),
)


In [ ]:
# sft_trainer_p1.train()

In [ ]:
from transformers import DataCollatorForLanguageModeling
import torch

class UnifiedCompletionCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer, *args, mlm=False, **kwargs)
        self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)

    def torch_call(self, examples):
        # Filter out 'text' strings to prevent the ValueError
        non_string_examples = []
        for e in examples:
            non_string_examples.append({k: v for k, v in e.items() if k != "text"})
            
        # Let the parent handle the tensor conversion/padding
        batch = super().torch_call(non_string_examples)
        
        # Apply the mask to 'labels'
        for i in range(len(batch["labels"])):
            curr_labels = batch["labels"][i]
            # Find sequence of token IDs
            for idx in range(len(curr_labels) - len(self.response_token_ids)):
                if curr_labels[idx : idx + len(self.response_token_ids)].tolist() == self.response_token_ids:
                    # Mask everything UP TO the end of the assistant header
                    batch["labels"][i][:idx + len(self.response_token_ids)] = -100
                    break
        return batch


# Initialize
collator = UnifiedCompletionCollator(
    response_template="<|im_start|>assistant\n", 
    tokenizer=tokenizer
)

In [ ]:
# Test the collator on 1 sample
sample = [train_dataset[0]]
test_batch = collator.torch_call(sample)

# Check how many tokens are NOT masked (-100)
labels = test_batch["labels"][0]
unmasked_count = (labels != -100).sum().item()
total_count = len(labels)

print(f"Total tokens: {total_count}")
print(f"Unmasked tokens (Assistant only): {unmasked_count}")

# Decode the unmasked part to see if it's the <t> block
unmasked_tokens = labels[labels != -100]
print("What the model is learning:")
print(tokenizer.decode(unmasked_tokens))

In [ ]:
sft_trainer_p2 = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,    
    data_collator=collator,
    dataset_text_field = "text",
    # formatting_func = format_sft_custom,
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    # dataset_batched=True,
    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 0,
        num_train_epochs = 6,
        learning_rate = 2e-4,
        logging_steps = 5,
        # save_steps = 15,
        # save_total_limit=3,
        # eval_strategy="steps",  # Tell the trainer to evaluate periodically
        # eval_steps=10,                # Run evaluation every 10 steps
        # metric_for_best_model="loss", # Or "eval_loss" (lower is better)
        # greater_is_better=False,
        # load_best_model_at_end=True,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/mnt/data/outputs/",
        report_to = "none",
        # completion_only_loss=True,
        # save_strategy="steps"
    ),
)


In [ ]:
sft_trainer_p2.train()
# sft_trainer.evaluate()

In [16]:
from transformers import TextStreamer
import time
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

tokenizer.chat_template = """{% for message in messages %}
<|im_start|>{{ message.role }}
{{ message.content }}<|im_end|>
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{%- endif %}"""

# --- CONFIGURATION ---
SYSTEM_PROMPT_TEMPLATE = """
INTENT_DETECTOR

{context_data}

"""

import json

CONTEXT_DATA = """{
  "PLANS": [
    {
      "p01": "maxis basic",
      "type": "N",
      "features": ["f01", "f03", "f04"]
    },
    {
      "p02": "celcom plus",
      "type": "N",
      "features": ["f01", "f05", "f07"]
    },
    {
      "p03": "Astro high speed",
      "type": "N",
      "features": ["f02", "f06", "f07"]
    },
    {
      "p04": Ultra 5G",
      "type": "N",
      "features": ["f02", "f06", "f08"]
    },
    {
      "p05": "Maxis Unlimited",
      "type": "N",
      "features": ["f02", "f09", "f10"]
    },
    {
      "p06": "Maxis SMB",
      "type": "B",
      "features": ["f02", "f09", "f11", "f12"]
    },
    {
      "p07": "Student Essentilas",
      "type": "S",
      "features": ["f01", "f13"]
    }
  ],
  "DEVICES": [
    {
      "d01": "Samsung Mini",
      "eligible_plans": ["p03", "p04", "p05", "p06"],
      "features": ["f15", "f16"]
    },
    {
      "d02": "iPhone Pro",
      "eligible_plans": ["p04", "p05", "p06"],
      "features": ["f15", "f17", "f18"]
    },
    {
      "d03": "Smart Extender",
      "eligible_plans": ["p04", "p05", "p06"],
      "features": ["f17", "f19"]
    },
    {
      "d04": " SIM Box",
      "eligible_plans": ["p01", "p02"],
      "features": ["f20"]
    },
    {
      "d05": "Simple 5G Dongle",
      "eligible_plans": ["p01", "p02", "p03", "p04", "p05", "p06", "p07", "p08"],
      "features": ["f02", "f21", "f22"]
    }
  ],
  "PROMOTIONS": [
    {
      "r01": "Premium Plan Boost",
      "eligible_plans": ["p04", "p05"]
    },
    {
      "r02": "Hello Saver",
      "eligible_plans": ["p01", "p02", "p03", "p04", "p05", "p06", "p07", "p08"]
    }
  ],
  "FEATURES": {
    "f01": { "name": "4g" },
    "f02": { "name": "5g" },
    "f03": { "name": "basic_voice" },
    "f04": { "name": "basic_sms" },
    "f05": { "name": "unlimited_sms" },
    "f06": { "name": "unlimited_voice" },
    "f07": { "name": "hotspot" },
    "f08": { "name": "entertainment_pack" },
    "f09": { "name": "priority_data" },
    "f10": { "name": "international_roaming" },
    "f11": { "name": "sla_support" },
    "f12": { "name": "vpn" },
    "f13": { "name": "social_pass" },
    "f14": { "name": "priority_care" },
    "f15": { "name": "wifi6" },
    "f16": { "name": "dual_band" },
    "f17": { "name": "mesh" },
    "f18": { "name": "parental_controls" },
    "f19": { "name": "plug_and_play" },
    "f20": { "name": "4g_router" },
    "f21": { "name": "usb_c" },
    "f22": { "name": "fallback_4g" }
  }
}

{
  "PROFILE": {
    "S": "p07"
  }
}

"""

# Define your Evaluation Questions
QUESTIONS = [
    # 1. Boundary Logic Trap (Target: LOGIC_TRAP)
    # Tests strictly > 60 vs >= 60 logic.
    "i want a plan with unlimited data.",
    "how much I am paying now?",
    "can i move to SMB plan along with 5G Dongle",
    "what all features my plan has",
    "compare tyat max basic and the student one ",
    "how m plans whose price under 176",
    "where i can buy an apples",
    "i want a sim box",
    "I want to buy sim box with hello discounts"
   
]


OUTPUT_FILE = "/mnt/data/training_data/evaluation_results.jsonl"

# --- RUN EVALUATION ---
with open(OUTPUT_FILE, "w") as f:
       
        for question in QUESTIONS:
            # 1. Prepare Prompt
            
            full_instruction = SYSTEM_PROMPT_TEMPLATE.format(
                context_data=CONTEXT_DATA,
                history_thinking_blocks="" # For future use when we implement HISTORY resolution logic
            )
            messages = [
                {"role": "system", "content": full_instruction.replace("{{", "{"). replace("}}","}")},
                {"role": "user", "content": question}
            ]

            # 2. Tokenize and Generate
            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to("cuda")

            # Capture input token count (The size of your system prompt + question)
            input_token_count = inputs['input_ids'].shape[1]

            # Start the timer
            start_time = time.time()

            # Execute model generation
            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=0.0,
                eos_token_id=tokenizer.eos_token_id
            )

            # Calculate total response time
            total_response_time = time.time() - start_time
            
            # 3. Decode Response and Capture Output Token Count
            # Slicing the output [len(inputs[0]):] ensures we only count NEW tokens
            generated_tokens = outputs[0][len(inputs[0]):]
            output_token_count = len(generated_tokens)
            
            response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            print(f"Q: {question}\nA: {response_text[:100]}...") # Console preview

            # 4. Save to JSONL with Performance Metrics
            eval_entry = {
                
                "question": question,
                "model_response": response_text,
                "metrics": {
                    "input_tokens": int(input_token_count),
                    "output_tokens": int(output_token_count),
                    "total_latency_sec": round(total_response_time, 4),
                    "tokens_per_sec": round(output_token_count / total_response_time, 2) if total_response_time > 0 else 0
                }
            }
            f.write(json.dumps(eval_entry) + "\n")

print(f"\nEvaluation Complete. Results saved to {OUTPUT_FILE}")

Q: i want a plan with unlimited data.
A: ?(p,p.GB=UNLIGB)[0]...
Q: how much I am paying now?
A: p07.$...
Q: can i move to SMB plan along with 5G Dongle
A: p07→p07+d01)...
Q: what all features my plan has
A: p07.F...
Q: compare tyat max basic and the student one 
A: ∷p07∷p07...
Q: how m plans whose price under 176
A: ?(p,p.$<176)[*]...
Q: where i can buy an apples
A: NIC...
Q: i want a sim box
A: p07→p07+SIM...
Q: I want to buy sim box with hello discounts
A: p07→p07+d01+r02...

Evaluation Complete. Results saved to /mnt/data/training_data/evaluation_results.jsonl


In [ ]:
//////

In [17]:
model_2, tokenizer_2 = FastLanguageModel.from_pretrained(
    model_name = "/mnt/data/outputs/checkpoint-84",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.044 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 274.18it/s, Materializing param=model.norm.weight]                              
/mnt/data/envs/llm_finetune_unsloth/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


RuntimeError: Error(s) in loading state_dict for PeftModelForCausalLM:
	size mismatch for base_model.model.model.embed_tokens.original_module.weight: copying a param with shape torch.Size([151739, 1536]) from checkpoint, the shape in current model is torch.Size([151936, 1536]).
	size mismatch for base_model.model.model.embed_tokens.modules_to_save.default.weight: copying a param with shape torch.Size([151739, 1536]) from checkpoint, the shape in current model is torch.Size([151936, 1536]).
	size mismatch for base_model.model.lm_head.original_module.weight: copying a param with shape torch.Size([151739, 1536]) from checkpoint, the shape in current model is torch.Size([151936, 1536]).
	size mismatch for base_model.model.lm_head.modules_to_save.default.weight: copying a param with shape torch.Size([151739, 1536]) from checkpoint, the shape in current model is torch.Size([151936, 1536]).

In [21]:
print(len(tokenizer_2))
print(model_2.config.vocab_size)

NameError: name 'tokenizer_2' is not defined

In [ ]:
model.save_pretrained("/mnt/data/models/amaiz_sales-llm-1.5B_adap")
tokenizer.save_pretrained("/mnt/data/models/amaiz_sales-llm-1.5B_adap")

In [ ]:
# model.save_pretrained_gguf("/mnt/data/models/amaiz_sales-llm-1.5B_gguf", tokenizer, quantization_method = "q4_k_m")

In [ ]:
# for name, param in model.named_parameters():
#     if "lora" in name:
#         if not param.requires_grad:
#             print(f"🚨 WARNING: {name} is FROZEN! Training will not work.")
#         else:
#             print(f"✅ {name} is trainable.")
#         break